In [1]:
!pip install faker

  Using cached faker-37.6.0-py3-none-any.whl.metadata (15 kB)
Using cached faker-37.6.0-py3-none-any.whl (1.9 MB)


In [76]:
import pandas as pd
import numpy as np
import random
from faker import Faker

fake = Faker('ko_KR')
np.random.seed(42)
random.seed(42)

# -----------------------------
# 1. 기본 설정
# -----------------------------
n_rows = 5000

# 날짜 (2023-01 ~ 2025-12)
date_range = pd.date_range("2023-01-01", "2025-12-31", freq="D")
dates = np.random.choice(date_range, n_rows)

# 매장 설정 (5개, 규모 다르게)
stores = [
    {"매장명": "서울점", "매장코드": "S001", "규모": 1.5},
    {"매장명": "부산점", "매장코드": "S002", "규모": 1.2},
    {"매장명": "대구점", "매장코드": "S003", "규모": 1.0},
    {"매장명": "광주점", "매장코드": "S004", "규모": 0.8},
    {"매장명": "원주점", "매장코드": "S005", "규모": 0.6},
]
store_choices = np.random.choice(stores, n_rows, p=[0.25,0.25,0.2,0.15,0.15])

# -----------------------------
# 2. 고객 설정
# -----------------------------
n_customers = 100
customers = []
for i in range(n_customers):
    gender = random.choice(["M","F"])
    age = np.random.randint(5, 70)
    customers.append({
        "고객코드": f"C{str(i+1).zfill(3)}",
        "성별": gender,
        "나이": age
    })

# 고객별 거래 특성
customer_behavior = {}
for c in customers:
    freq = np.random.choice(["high","medium","low"], p=[0.3,0.5,0.2])
    active_period = np.random.choice(["early","mid","late"], p=[0.3,0.4,0.3])
    amount_level = np.random.choice(["high","low"], p=[0.3,0.7])
    product_variety = np.random.choice(["wide","narrow"], p=[0.4,0.6])
    customer_behavior[c["고객코드"]] = {
        "freq": freq, 
        "active_period": active_period, 
        "amount_level": amount_level, 
        "product_variety": product_variety
    }

customer_choices = np.random.choice(customers, n_rows)

# -----------------------------
# 3. 티니핑 스타일 상품 생성기 (카테고리별 가격대 반영)
# -----------------------------
adjectives = ["반짝", "꿈꾸는", "별빛", "달콤", "용기", "마법", "해피", "포근"]
characters = ["하트핑", "드림핑", "스위티핑", "버블핑", "러블리핑", "스타핑", "큐티핑", "조이핑"]
categories = ["라면", "쿠키", "주스", "빵", "아이스크림", "초콜릿", "사탕", "우유", "쥬얼리", "토이"]

price_ranges = {
    "라면": (3000, 6000),
    "쿠키": (1000, 5000),
    "주스": (1500, 4500),
    "빵": (2000, 6000),
    "아이스크림": (1000, 3000),
    "초콜릿": (1500, 5000),
    "사탕": (1000, 3000),
    "우유": (1500, 4000),
    "쥬얼리": (5000, 30000),
    "토이": (5000, 30000)
}

products = []
for i in range(20):
    adj = random.choice(adjectives)
    char = random.choice(characters)
    cat = random.choice(categories)
    name = f"{adj}{char} {cat}"
    low, high = price_ranges[cat]
    price = np.random.randint(low//100, high//100 + 1) * 100  # 100단위 맞춤
    products.append({
        "상품코드": f"P{str(i+1).zfill(3)}",
        "상품명": name,
        "판매단가": price
    })

product_choices = np.random.choice(products, n_rows)

# -----------------------------
# 4. 구매수량 및 할인율
# -----------------------------
quantities = []
discounts = []
for i in range(n_rows):
    store = store_choices[i]
    cust = customer_choices[i]
    
    base_qty = np.random.poisson(2)
    if customer_behavior[cust["고객코드"]]["freq"] == "high":
        base_qty += np.random.poisson(2)
    qty = max(1, int(base_qty * store["규모"]))
    quantities.append(qty)
    
    base_discount = np.random.uniform(0,0.1)
    if customer_behavior[cust["고객코드"]]["amount_level"] == "high":
        base_discount += np.random.uniform(0,0.05)
    discounts.append(round(min(base_discount,0.15),2))

# -----------------------------
# 5. 데이터프레임 생성
# -----------------------------
df = pd.DataFrame({
    "날짜": dates,
    "매장명": [s["매장명"] for s in store_choices],
    "매장코드": [s["매장코드"] for s in store_choices],
    "고객코드": [c["고객코드"] for c in customer_choices],
    "성별": [c["성별"] for c in customer_choices],
    "나이": [c["나이"] for c in customer_choices],
    "상품코드": [p["상품코드"] for p in product_choices],
    "상품명": [p["상품명"] for p in product_choices],
    "판매단가": [p["판매단가"] for p in product_choices],
    "구매수량": quantities,
    "할인율": discounts
})

df


,날짜,매장명,매장코드,고객코드,성별,나이,상품코드,상품명,판매단가,구매수량,할인율
0,2025-05-10,부산점,S002,C056,F,11,P018,포근드림핑 빵,5100,1,0.03
1,2025-12-31,대구점,S003,C030,M,13,P007,달콤스위티핑 초콜릿,3500,1,0.08
2,2025-11-10,대구점,S003,C022,M,38,P006,별빛드림핑 아이스크림,1700,2,0.09
3,2023-05-02,원주점,S005,C075,M,6,P011,달콤하트핑 빵,4500,1,0.05
4,2024-04-11,원주점,S005,C081,F,17,P008,별빛하트핑 토이,19500,1,0.01
...,...,...,...,...,...,...,...,...,...,...,...
4995,2023-01-13,광주점,S004,C022,M,38,P014,포근스위티핑 아이스크림,1600,4,0.14
4996,2025-11-07,부산점,S002,C030,M,13,P018,포근드림핑 빵,5100,2,0.06
4997,2024-11-24,서울점,S001,C062,M,31,P002,용기하트핑 쿠키,2700,3,0.06
4998,2025-08-30,원주점,S005,C073,F,20,P001,해피조이핑 쥬얼리,8400,1,0.01


In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   날짜      5000 non-null   datetime64[ns]
 1   매장명     5000 non-null   object        
 2   매장코드    5000 non-null   object        
 3   고객코드    5000 non-null   object        
 4   성별      5000 non-null   object        
 5   나이      5000 non-null   int64         
 6   상품코드    5000 non-null   object        
 7   상품명     5000 non-null   object        
 8   판매단가    5000 non-null   int64         
 9   구매수량    5000 non-null   int64         
 10  할인율     5000 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(6)
memory usage: 429.8+ KB


In [80]:
df.groupby('상품명')['판매단가'].agg(['sum', 'mean', 'median','std'])

,sum,mean,median,std
상품명,,,,
꿈꾸는드림핑 우유,812000,3500.0,3500.0,0.0
꿈꾸는스위티핑 주스,637000,2600.0,2600.0,0.0
꿈꾸는스타핑 아이스크림,652800,2400.0,2400.0,0.0
달콤드림핑 초콜릿,704700,2700.0,2700.0,0.0
달콤러블리핑 사탕,467200,1600.0,1600.0,0.0
달콤스위티핑 초콜릿,906500,3500.0,3500.0,0.0
달콤하트핑 빵,1116000,4500.0,4500.0,0.0
마법조이핑 라면,1036300,4300.0,4300.0,0.0
마법조이핑 쥬얼리,7043200,28400.0,28400.0,0.0


In [82]:
df.to_csv('sales_data.csv')